# 🔬 SGLang 源码级深度剖析

> **核心命题**：如果把前缀缓存做到极致会怎样？RadixAttention 的答案——自动发现和复用任何公共前缀。

SGLang 是 2024 年才出现的新玩家，但它的 **RadixAttention** 创新直接改变了推理服务的格局。
vLLM 后来也借鉴了它的思路来改进自己的 Prefix Caching。

## SGLang = 编程语言 + 推理引擎

SGLang 有双重身份：
1. **SGLang DSL**：一种描述 LLM 调用逻辑的领域特定语言（类似 guidance / LMQL）
2. **SGLang Runtime**：基于 RadixAttention 的高性能推理引擎

```
┌──────────────────────────────────────────────────────────────┐
│                       SGLang 架构                            │
├──────────────────────────────────────────────────────────────┤
│                                                               │
│  ┌─────────────────────────────────────────────────────────┐ │
│  │                SGLang DSL (前端)                         │ │
│  │  • Python 嵌入式 DSL: sgl.gen(), sgl.select(), ...      │ │
│  │  • 自动并行：fork() + join() 模式                       │ │
│  │  • 结构化输出约束：regex / JSON schema                   │ │
│  └─────────────────────────┬───────────────────────────────┘ │
│                            │                                  │
│  ┌─────────────────────────▼───────────────────────────────┐ │
│  │              SGLang Runtime (后端)                       │ │
│  │  • RadixAttention — 基于 radix tree 的前缀缓存           │ │
│  │  • Tokenizer — 高效的 tokenization                       │ │
│  │  • Scheduler — Continuous batching + prefix-aware        │ │
│  │  • Model Runner — 模型执行（支持 vLLM 兼容后端）         │ │
│  └─────────────────────────────────────────────────────────┘ │
│                                                               │
└──────────────────────────────────────────────────────────────┘
```

## 1. RadixAttention：前缀缓存的终极形态

这是 SGLang 最核心的创新。vLLM 的 Prefix Caching 需要显式指定前缀，RadixAttention 自动发现。

### 1.1 数据结构：Radix Tree (Trie)

```
Radix Tree (压缩前缀树) 的例子:

假设已经缓存了以下序列的 KV Cache：
  - "You are a helpful assistant"
  - "You are a coding expert"
  - "Hello world"

Radix Tree:
                    root
                   /    \
            "Hello "    "You are a "
              |           /        \
         "world"   "helpful "    "coding "
                     |              |
               "assistant"      "expert"

每个节点存储:
  - token_sequence: 该边的 token 序列
  - kv_blocks: 对应的物理 KV Cache blocks (共享!)
  - ref_count: 被多少个请求引用
  - last_access_time: LRU 淘汰

当新请求 "You are a helpful assistant, tell me about Python" 到来:
  1. 在 radix tree 中匹配: root → "You are a " → "helpful " → "assistant"
  2. 匹配到 7 个 token 的前缀 ← 自动发现！不需要手动声明
  3. 这 7 个 token 的 KV Cache 直接复用
  4. 只需要 prefill ", tell me about Python" 这 5 个新 token
```

### 1.2 vs vLLM Prefix Caching

```
场景: 3 个请求共享不同长度的 system prompt

Request A: "You are a helpful assistant" + "What is the weather?"
Request B: "You are a helpful assistant" + "Write a poem"
Request C: "You are a helpful assistant, especially good at math" + "Solve 2+2"

vLLM Prefix Caching (hash-based):
  - Rq A & B 共享: hash("You are a helpful assistant") → 命中 (8 tokens)
  - Rq C: hash("You are a helpful assistant, especially...") → 完全不同的 hash → 未命中！
  ← vLLM 必须 hash 完全匹配，C 的前 8 个 token 虽然和 A/B 一样但没被复用

SGLang RadixAttention (trie-based):
  - Rq A: 插入路径 root→"You are a helpful assistant"
  - Rq B: 完全匹配路径 → 命中 8 tokens
  - Rq C: 路径 root→"You are a helpful " 匹配！然后分叉出 "assistant, especially..."
  ← C 的前 6 个 token 和 A/B 共享（尽管后面不同）
  ← 这就是 radix tree 的优势：最长公共前缀自动匹配
```

## 2. SGLang DSL：编程语言视角

SGLang 将 LLM 调用建模为编程语言原语。

```python
import sglang as sgl

# SGLang 的核心原语
@sgl.function
def multi_turn_chat(s):
    # gen: 生成文本
    s += sgl.system("You are a helpful assistant.")
    
    # fork: 并行探索多个分支（共享前缀！）
    s += sgl.user("What should I eat for dinner?")
    
    # 并行 fork — 共享 user 消息的 KV Cache
    forks = s.fork(3)
    forks[0] += sgl.assistant(sgl.gen("option1", max_tokens=100))
    forks[1] += sgl.assistant(sgl.gen("option2", max_tokens=100))
    forks[2] += sgl.assistant(sgl.gen("option3", max_tokens=100))
    forks.join()
    
    # select: 约束选择
    s += sgl.user("Which option is healthiest?")
    s += sgl.assistant(sgl.select(["Option 1", "Option 2", "Option 3"]))

# SGLang 编译为对 Runtime 的调用
# fork → 3 个分支共享前缀 KV Cache, 各分支独立生成
# join → 等待所有分支完成
```

**DSL 的价值**：用 fork/join 显式表达并行意图，Runtime 自动优化为共享 KV Cache。

## 3. 结构化生成（Structured Generation）

SGLang 对结构化输出的支持是目前最成熟的。

```python
# JSON 约束生成
@sgl.function
def extract_person(s, text):
    s += sgl.user(f"Extract from: {text}")
    s += sgl.assistant(
        sgl.gen("result", 
                max_tokens=200,
                regex=r'\{\s*"name":\s*"[^"]+",\s*"age":\s*\d+\s*\}')
    )

# 实现原理: 将 regex 编译为 FSM (有限状态机)
# 每个推理 step:
#   1. 获取当前 FSM 允许的 token 集合 (mask)
#   2. 将 logits 中不允许的 token 设为 -inf
#   3. 采样 → 保证输出符合 regex
#   4. FSM 状态转移

# 编译示例:
# regex: \{"name": "[^"]+", "age": \d+\}
# FSM 会保证:
#   Step 1: 只能生成 '{'
#   Step 2: 只能生成 '"'
#   Step 3: 只能生成 'n'
#   Step 4: 只能生成 'a'
#   ...
#   在 value 区域: 除了 '"' 以外的所有字符都允许
#   在 age 区域: 只有 '0'-'9' 允许
```

SGLang 的 regex → FSM 编译器用 `outlines` 库，但在 SGLang Runtime 层做了大量优化（FSM 状态缓存、mask 计算向量化）。

## 4. 性能分析：为什么 RadixAttention 在各种场景下几乎总是赢？

```
场景 1: Few-shot prompting (最有利场景)
  每个请求都有相同的 N 个 example → Radix tree 完美匹配
  前缀匹配率: 90%+ → prefill 计算节省 90%
  
场景 2: Multi-turn chat
  每轮对话的历史是公共前缀
  前缀匹配率: 60-80% → prefill 节省显著
  
场景 3: Parallel sampling ("generate 5 responses and pick the best")
  fork(5) → 5 个分支共享 prompt 的 KV Cache
  vLLM 也能做但需要手动管理，SGLang 自动

场景 4: Beam search
  每个 beam 的大部分前缀相同 → Radix tree 自然支持
```

**唯一劣势的场景**：完全随机、无公共前缀的请求 → Radix tree 退化为线性链表，有轻微内存开销。

## 5. SGLang vs vLLM

| 维度 | SGLang | vLLM |
|------|--------|------|
| **核心创新** | RadixAttention (trie-based prefix caching) | PagedAttention (block-based KV management) |
| **出现时间** | 2024 Q1 | 2023 Q2 |
| **前缀缓存** | 自动发现，trie 匹配 | hash-based，需精确匹配 |
| **编程模型** | SGLang DSL (fork/join/select) | OpenAI API only |
| **结构化生成** | 最好（regex FSM 原生集成） | 通过 guided_decoding 支持 |
| **性能（一般）** | 与 vLLM 持平 | — |
| **性能（前缀多）** | 明显优于 vLLM | — |
| **生态** | 较小但增长快 | 最大 |
| **部署** | pip install | pip install |

SGLang 和 vLLM 不是替代关系，而是互补：
- 用 vLLM 搭建通用 API 服务
- 用 SGLang 做需要复杂 LLM 编程逻辑的应用（multi-agent、parallel sampling、structured output）

## 下一步

- → `07-ray-serve-llm.ipynb`：分布式推理的终极方案
- → 返回 `../00-overview.ipynb` 查看全景对比